# Example & Explanations - Privacy evaluation based on adversarial testing with TAPAS.
This file a simple pipeline using TAPAS. It allows to play with the different components on a lighweight example.
It also contains explanation cells to give context and instruction over the execution steps.
This notebook focus on the 2011 UK Census Microdata (https://www.ons.gov.uk/census/2011census/2011censusdata/censusmicrodata), a dataset which contains information from individual census responses (treated to protect confidentiality) allowing comparison of characteristics. This dataset is used in the TAPAS and Achille's Heel dataset for privacy evaluation purpose. For more information about the variables, see https://www.ons.gov.uk/census/2011census/2011censusdata/censusmicrodata/microdatateachingfile/variablelist. 

## Imports and other setup

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd

import tapas.datasets
import tapas.generators
import tapas.threat_models
import tapas.attacks
import tapas.report

In [2]:
os.chdir('..')

In [3]:
DATA_FOLDER = Path('./data')
OUTPUT_FOLDER = Path('./generated')
OUTPUT_FOLDER_EDA = OUTPUT_FOLDER/'eda'
OUTPUT_FOLDER_GENERATOR = OUTPUT_FOLDER/'datasets' # where to store the generator (not related to privacy) generated datasets
OUTPUT_FOLDER_PRIVACY = OUTPUT_FOLDER/'privacy_evaluation'

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

/var/folders/mb/y8_byc0x1fl80yvxkf612cmm0000gp/T/ipykernel_17667/1932600153.py:9: FutureWarning: The pandas.np module is deprecated and will be removed from pandas in a future version. Import numpy directly instead.
  pd.np.random.seed(RANDOM_STATE)


In [4]:
def columns_as_category(df, cat_columns):
    df = df.copy()
    for col in cat_columns:
        df[col] = df[col].astype('category')    
    return df

## Privacy evaluation

### Define the dataset and the generator to evaluate

In [5]:
from tools.tapas.tapas_data_processors import AdultDataProcessor
adult = tapas.datasets.TabularDataset.read(DATA_FOLDER/"adult", 'adult')
adult = AdultDataProcessor.process_tapas_tabulardataset(adult)

# drop target record
target_record_indices = [39435]
target_record = adult.get_records(target_record_indices)
adult.drop_records(target_record_indices, in_place=True)
adult.data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 45221 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             45221 non-null  int64  
 1   workclass       45221 non-null  object 
 2   fnlwgt          45221 non-null  float64
 3   education       45221 non-null  object 
 4   education-num   45221 non-null  int64  
 5   marital-status  45221 non-null  object 
 6   occupation      45221 non-null  object 
 7   relationship    45221 non-null  object 
 8   race            45221 non-null  object 
 9   sex             45221 non-null  object 
 10  capital-gain    45221 non-null  float64
 11  capital-loss    45221 non-null  float64
 12  hours-per-week  45221 non-null  float64
 13  native-country  45221 non-null  object 
 14  income          45221 non-null  object 
dtypes: float64(4), int64(2), object(9)
memory usage: 5.5+ MB


### Define the attack
Define how the attacker will proceed to perform its membership inference attack. In this first case, we define a GroundHog attack: infer multiple datasets and extract their statistics as features for classification.  

In [6]:
raw_gen = tapas.generators.Raw()
raw_gen.fit(adult)
raw_output = raw_gen.generate(num_samples=10)

In [7]:
raw_output.data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 10 entries, 17957 to 27965
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             10 non-null     int64  
 1   workclass       10 non-null     object 
 2   fnlwgt          10 non-null     float64
 3   education       10 non-null     object 
 4   education-num   10 non-null     int64  
 5   marital-status  10 non-null     object 
 6   occupation      10 non-null     object 
 7   relationship    10 non-null     object 
 8   race            10 non-null     object 
 9   sex             10 non-null     object 
 10  capital-gain    10 non-null     float64
 11  capital-loss    10 non-null     float64
 12  hours-per-week  10 non-null     float64
 13  native-country  10 non-null     object 
 14  income          10 non-null     object 
dtypes: float64(4), int64(2), object(9)
memory usage: 1.2+ KB


In [8]:
cat_cols = adult.description.one_hot_cols
# compute the total number of different cateogories within all categorical columns
total_categories = sum([adult.data[col].nunique() for col in cat_cols])
# compute the total of numerical columns
total_numerical = sum([1 for col in adult.data.columns if col not in cat_cols])
total_categories+total_numerical

106

In [11]:
from tapas.generators import ReprosynGenerator
from reprosyn.methods import DS_BAYNET
baynet_gen = ReprosynGenerator(DS_BAYNET, seed=42)
baynet_gen.fit(adult.sample(10, random_state=42))
baynet_output = baynet_gen.generate(10)
baynet_output.data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             10 non-null     int64  
 1   workclass       10 non-null     object 
 2   fnlwgt          10 non-null     float64
 3   education       10 non-null     object 
 4   education-num   10 non-null     int64  
 5   marital-status  10 non-null     object 
 6   occupation      10 non-null     object 
 7   relationship    10 non-null     object 
 8   race            10 non-null     object 
 9   sex             10 non-null     object 
 10  capital-gain    10 non-null     float64
 11  capital-loss    10 non-null     float64
 12  hours-per-week  10 non-null     float64
 13  native-country  10 non-null     object 
 14  income          10 non-null     object 
dtypes: float64(4), int64(2), object(9)
memory usage: 1.3+ KB


In [ ]:
sys.path.append('..')

In [21]:
np.random.seed(RANDOM_STATE)
pd.np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
# ddpm_gen = SynthcityGenerator("ddpm", "TabDDPM", 42, metadata=adult.description.schema, 
#                               n_iter=10)
hidden_baynet_gen = ReprosynGenerator(DS_BAYNET, seed=RANDOM_STATE)
hidden_baynet_gen.fit(adult.sample(2000, random_state=RANDOM_STATE))
published_baynet_output = baynet_gen.generate(2000)

# generator = baynet_gen

# raw_gen = tapas.generators.Raw()
# raw_gen.fit(baynet_output)
# raw_output = raw_gen.generate(num_samples=1000)
gen = ReprosynGenerator(DS_BAYNET, seed=RANDOM_STATE)

data_knowledge = tapas.threat_models.AuxiliaryDataKnowledge(
    published_baynet_output,
    auxiliary_split=0.5,
    num_training_records=1000
)

# num
from tapas.threat_models import NoBoxKnowledge
sdg_knowledge_raw = tapas.threat_models.BlackBoxKnowledge(
    gen,
    num_synthetic_records=1000
)
# sdg_knowledge_raw = tapas.threat_models.BlackBoxKnowledge(
#     gen,
#     num_synthetic_records=1000
# )

from tqdm import tqdm
# the kind of attack seems to take care of the actual attack performed.
threat_model = tapas.threat_models.TargetedMIA(
    attacker_knowledge_data=data_knowledge,
    attacker_knowledge_generator=sdg_knowledge_raw,
    target_record=target_record,
    iterator_tracker=tqdm,
    generate_pairs=True,
    replace_target=True
)

from tapas.attacks.groundhog import GroundhogAttack
from tapas.attacks.closest_distance import ClosestDistanceMIA
attack = ClosestDistanceMIA()
# attack = GroundhogAttack(use_naive=True, use_hist=False, use_corr=False)
attack.train(threat_model, num_samples=10)
attack_summary = threat_model.test(attack, num_samples=10)
attack_summary.get_metrics()

/var/folders/mb/y8_byc0x1fl80yvxkf612cmm0000gp/T/ipykernel_17667/1959794975.py:2: FutureWarning: The pandas.np module is deprecated and will be removed from pandas in a future version. Import numpy directly instead.
  pd.np.random.seed(RANDOM_STATE)






















100%|██████████| 10/10 [00:10<00:00,  1.00s/it]






















100%|██████████| 10/10 [00:10<00:00,  1.01s/it]


,dataset,target_id,generator,attack,accuracy,true_positive_rate,false_positive_rate,mia_advantage,privacy_gain,auc,effective_epsilon
0,adult (AUX),39435,<class 'reprosyn.methods.data_synthesiser.wrap...,"ClosestDistance(Hamming, accuracy)",1.0,1.0,0.0,1.0,0.0,1.0,inf
